In this notebook, we will show how to conduct energy simulation for an air-cooled data center with 5 data halls and a chiller plant with 5 chillers. In this case, the control variables is the chilled water supply temperature.

In [1]:
import numpy as np
from dctwin.interfaces.gym_env import EPlusEnv
from dctwin.utils import config as env_config
from google.protobuf import json_format

from dctwin.utils import setup_logging, read_engine_config

(1) Setup environment variables

In [2]:
engine_config = "config.prototxt"
env_config.eplus.engine_config_file = engine_config

(2) Read configuration files

In [3]:
config = read_engine_config(engine_config=engine_config)
setup_logging(config.logging_config, engine_config=engine_config)

2022-11-07 15:46:23.948 | INFO     | dctwin.utils.config:setup_logging:179 - Logging to /Users/zhiweicao/Dropbox/Mac/Desktop/dctwin/tutorials/eplus/log/2022-11-07-15-46-23_eplus ...


(3) Build environment

In [4]:
env_config_name = config.WhichOneof("EnvConfig")
env_params = json_format.MessageToDict(
    getattr(config, env_config_name).env_params,
    preserving_proto_field_name=True,
)
env = EPlusEnv(
    config=getattr(config, env_config_name),
    reward_fn=None,
    schedule_fn=None,
    **env_params,
)

2022-11-07 15:46:23.954 | INFO     | dctwin.interfaces.gym_env.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc1_supply_temperature
2022-11-07 15:46:23.955 | INFO     | dctwin.interfaces.gym_env.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc2_supply_temperature
2022-11-07 15:46:23.956 | INFO     | dctwin.interfaces.gym_env.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc3_supply_temperature
2022-11-07 15:46:23.958 | INFO     | dctwin.interfaces.gym_env.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc4_supply_temperature
2022-11-07 15:46:23.958 | INFO     | dctwin.interfaces.gym_env.ds.scalars:__init__:51 - Fixed value 17.0 is set for dc5_supply_temperature
2022-11-07 15:46:23.959 | INFO     | dctwin.interfaces.gym_env.base_env:_set_simulation_time:100 - Using pre-set simulation time
2022-11-07 15:46:24.006 | INFO     | dctwin.backends.eplus.core:_set_up_socket:75 - socket listening on 0.0.0.0:49853


(4) Run EnergyPlus simulation

In [5]:
water_supply_sp = 12.0
env.reset()
done = False
while not done:
    act = np.array([water_supply_sp])
    obs, rew, done, truncated, info = env.step(act)

2022-11-07 15:46:24.013 | INFO     | dctwin.backends.eplus.core:_parse_idf_and_gen_bcvtb_config:111 - Parsing IDF file...
2022-11-07 15:46:25.442 | INFO     | dctwin.backends.eplus.core:_parse_idf_and_gen_bcvtb_config:123 - Generating BCVTB Config ...
2022-11-07 15:46:25.443 | INFO     | dctwin.backends.eplus.parser:save_cfg_xml:711 - no internal schedule config added
2022-11-07 15:46:25.515 | INFO     | dctwin.backends.core:run_container:65 - docker mount: /Users/zhiweicao/Dropbox/Mac/Desktop/dctwin/tutorials/eplus/log/2022-11-07-15-46-23_eplus/eplus_output/episode_1
2022-11-07 15:46:25.516 | INFO     | dctwin.backends.core:run_container:66 - docker run: /bin/bash -c /usr/local/EnergyPlus-9-5-0/energyplus -w SGP_Singapore.486980_IWEC.epw -r 5DC_4CH.idf
2022-11-07 15:46:31.483 | INFO     | dctwin.backends.eplus.core:_run_backend:176 - Got connection from ('127.0.0.1', 50100)
2022-11-07 15:46:51.903 | DEBUG    | dctwin.backends.eplus.core:receive_status:241 - Came to the end of one epis